# Imports

In [15]:
import pandas as pd

# Import and Process Dataframes

In [16]:
data = pd.read_csv("data/raw/videos.csv")
channel_data = pd.read_csv("data/processed/channels.csv")

As 2026 is not complete, we decided not to take into account this year to avoid bias.

In [17]:
data = data[~data["published_at"].str.startswith("2026")]

We added the title of the channel for better readability of the dataset.

In [18]:
data["channel_title"] = channel_data.set_index("channel_id").loc[data["channel_id"], "channel_title"].values

We noticed that Stanford data had many lecture videos, that were later on published on a new dedicated channel. As it was largely influencing our results, particularly regarding video format metrics, we decided to remove them from the dataset.

In [19]:
stanford_data_outliers = data.loc[
    (data["channel_title"] == "Stanford") &
    (data["title"].str.strip().str.lower().str.contains("lecture", na=False) | data["title"].str.contains(r"\d+\.\s", regex=True, na=False))
]

data = data.drop(stanford_data_outliers.index)

We then categorized the channel type (Independant / Institutional).

In [20]:
data["Institution"] = data["channel_id"].isin(channel_data[channel_data["group_label"] == "institutional"]["channel_id"])

Published dates were parsed into datetime format and the publication year was extracted into a separate column for simplicity.

In [21]:
data["published_at"] = pd.to_datetime(data["published_at"], utc=True, errors="coerce")
data["published_year"] = data["published_at"].dt.year

# Export

In [22]:
data.to_csv("data/processed/videos_cleaned.csv", index=False)